# chessplan

**What is the *plan* behind a move?** Pick two candidate moves from the same position. Force each one, let Stockfish play out the rest many times, and see which follow-up moves show up *sooner and more often* after A than after B.

Every downstream move gets a **relevance** score under each candidate:

> `relevance` = average of `1 / (how many of your own moves until you first play it)`, scoring **0** when you never play it.

So **1.0** = played immediately in every rollout, **0** = never played. Higher = more central to that plan. `delta = rel_A − rel_B` says which candidate owns the move.

In [ ]:
%load_ext autoreload
%autoreload 2
from chessplan import SetupBoard, analyze, show, to_frame

## 1. Set up the position

Click a piece, then its destination. **Undo** / **Reset** are under the board. You can also seed it from code with `.play("e4", "e5")` (SAN or UCI), or start from a FEN: `SetupBoard(chess.Board(fen))`.

In [ ]:
b = SetupBoard()
b.play("e4", "e5", "Nf3", "Nc6")   # the Ponziani position; edit or delete this line
b

## 2. Compare two candidate moves

| knob | meaning |
|---|---|
| `temp` | pawns. How loosely the engine explores — higher plays more second-best moves. **Changes what the numbers mean.** |
| `horizon` | plies rolled out. How far ahead a "plan" is allowed to reach. **Changes what the numbers mean.** |
| `n` | rollouts per candidate — more is just less noise |
| `depth` | engine search depth per move — cost |
| `multipv` | how many candidate moves the engine considers at each step, i.e. the pool the sampler draws from. `1` makes rollouts deterministic and ignores `temp`. |

`n=30, depth=6` takes about a minute. Drop to `n=10, depth=4` while exploring.

In [ ]:
rows, A, B = analyze(b.board, "c3", "Nc3", n=30, horizon=14, depth=6, temp=0.6, seed=0)
print(f"{len(rows)} downstream moves compared")

## 3. Read the plans

**Boards** — each candidate played (green arrow), with coloured arrows to the follow-ups most central to *that* plan; the destination shading is strongest for the highest-relevance move.

**Bars** — split into **A's plan**, **common to both**, and **B's plan**. Bar length is the relevance gap.

In [ ]:
show(b.board, rows, A, B)

## 4. The numbers

`rel_A` / `rel_B` are the relevance scores; `P_A` / `P_B` are how often the move appeared at all within the horizon. Sorted strongest-A-plan first, so the tail is B's plan.

In [ ]:
df = to_frame(rows)
df.head(20).style.bar(subset=["delta"], align="zero", color=["#e8734c", "#4c9be8"]) \
                 .format({"rel_A": "{:.3f}", "rel_B": "{:.3f}",
                          "P_A": "{:.2f}", "P_B": "{:.2f}", "delta": "{:+.3f}"})